In [ ]:
""" import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

from model import GATClassifier

# -----------------------------
# SETTINGS
# -----------------------------
BATCH_SIZE = 64
NUM_EPOCHS = 1
LR = 0.001
HIDDEN_CHANNELS = 64
NUM_CLASSES = 3  # Alzheimer's classes
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -----------------------------
# LOAD DATA
# -----------------------------
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# -----------------------------
# INITIALIZE MODEL
# -----------------------------
model = GATClassifier(
    in_channels=1,              # Feature: mean intensity
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=NUM_CLASSES
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = torch.nn.CrossEntropyLoss()

# -----------------------------
# TRAINING LOOP
# -----------------------------
def train():
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()

        out = model(batch.x, batch.edge_index, batch.batch)
        loss = loss_fn(out, batch.y.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = out.argmax(dim=1)
        correct += (preds == batch.y.view(-1)).sum().item()
        total += batch.num_graphs

    acc = correct / total
    return total_loss / len(train_loader), acc

# -----------------------------
# VALIDATION LOOP
# -----------------------------
def evaluate():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            out = model(batch.x, batch.edge_index, batch.batch)
            preds = out.argmax(dim=1)
            correct += (preds == batch.y.view(-1)).sum().item()
            total += batch.num_graphs

    acc = correct / total
    return acc

# -----------------------------
# RUN TRAINING
# -----------------------------
print("Training model...\n")
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train()
    val_acc = evaluate()
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

print("\n✅ Training complete.")
 """

In [ ]:
""" import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter
from torch_geometric.nn import GATConv, global_mean_pool


class BGANLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(BGANLayer, self).__init__()
        self.gat = GATConv(in_channels, out_channels, heads=1, concat=True)
        self.local_weight = Parameter(torch.randn(out_channels, out_channels))

    def forward(self, x, edge_index):
        # Apply graph attention
        x_att = self.gat(x, edge_index)
        # Local attention update
        out = F.relu(torch.matmul(x_att, self.local_weight) + x_att)
        return out


class GlobalAttention(nn.Module):
    def __init__(self, in_channels):
        super(GlobalAttention, self).__init__()
        self.linear = nn.Linear(in_channels, 1)

    def forward(self, x, batch):
        # x: [N_nodes, F]; batch: [N_nodes] with graph ids
        weights = torch.sigmoid(self.linear(x))
        x = x * weights
        x_pool = global_mean_pool(x, batch)  # [N_graphs, F]
        return x_pool


class BGANClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super(BGANClassifier, self).__init__()
        self.local_layer = BGANLayer(in_channels, hidden_channels)
        self.global_attention = GlobalAttention(hidden_channels)
        self.classifier = nn.Linear(hidden_channels, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.local_layer(x, edge_index)          # Local graph attention
        x = self.global_attention(x, batch)          # Global pooling with attention
        out = self.classifier(x)                     # Graph-level classification
        return out
 """

In [ ]:
""" import dgl
import numpy as np
import torch as th
import torch.nn as nn
import torch.nn.functional as F
import dgl.nn.pytorch as dglnn
import random
from torch.nn.parameter import Parameter
device=th.device('cuda' if th.cuda.is_available() else 'cpu')

class BGAN(nn.Module):
    def __init__(self,in_dim, out_dim, n_classes):
        super(BGAN, self).__init__()
        self.in_dim = in_dim
        self.classify = nn.Linear(out_dim, n_classes)
        self.GATLayer = GATLayer(in_dim,out_dim)
        self.conv = dglnn.GraphConv(in_dim, 1)
        self.localw= Parameter(th.randn(self.max_neighs+self.in_dim-1,out_dim))
        self.ReLU = nn.ReLU(inplace=True)
        self.Sigmoid = nn.Sigmoid()
        self.Softmax = nn.Softmax(dim=1)
    
    def reset_parameters(self):
        gain = nn.init.calculate_gain('relu')
        nn.init.xavier_normal_(self.localw, gain=gain)

    def get_graphs(self, block):
        g = block
        self.max_neighs = 10

        in_edges = th.tensor([]).to(device)
        out_edges = th.tensor([]).to(device)
        for node in range(len(g.dstnodes())): 
            src, dst = g.in_edges(int(node))
            if src.shape[0] == 0:
                src = th.tensor([int(node)], dtype=th.int64)
            src = src.repeat(self.max_neighs)[:self.max_neighs]
            dst = dst.repeat(self.max_neighs)[:self.max_neighs]
            out_edges = th.cat([out_edges,src]).to(device)
            in_edges = th.cat([in_edges,dst]).to(device)
        graph = dgl.graph((out_edges.long(),in_edges.long())).to(device)
        return graph

    def GlobalAttention(self,g,h):
        globalweight = self.Softmax(self.conv(g,h).reshape(self.conv(g,h).shape[0]//self.in_dim,self.in_dim))
        globalweight = globalweight.reshape(globalweight.shape[0]*self.in_dim,1)
        return globalweight

    def LocalAttention(self,g,h):
        graph = self.get_graphs(g)
        output = self.GATLayer(graph,h)
        updatefeat = self.ReLU(th.mm(output,self.localw) + h)
        return updatefeat
    
    def forward(self,g,h):
        '''Local Attention: Update features layers'''
        feats = self.LocalAttention(g,h)   
        '''Global Attention: Calculate weight layers'''
        weight = self.GlobalAttention(g,h)  
        
        '''Classifier'''
        updatafeat = weight * feats                       
        with g.local_scope(): 
            g.ndata['h'] = updatafeat
            hg = dgl.mean_nodes(g, 'h')
        return self.classify(hg)

class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super(GATLayer, self).__init__()
        self.fc = nn.Linear(in_dim, out_dim, bias=False)
        self.attn_fc = nn.Linear(2 * out_dim, 1, bias=False)
        self.conv = CNN()
        self.reset_parameters()
        
    def reset_parameters(self):
        gain = nn.init.calculate_gain('relu')
        nn.init.xavier_normal_(self.fc.weight, gain=gain)
        nn.init.xavier_normal_(self.attn_fc.weight, gain=gain)

    def edge_attention(self, edges):
        z2 = th.cat([edges.src['z'], edges.dst['z']], dim=1)
        a = self.attn_fc(z2)
        return {'e': F.leaky_relu(a)}
    def message_func(self, edges):
        return {'z': edges.src['z'], 'e': edges.data['e']}
    def reduce_func(self, nodes):
        alpha = F.softmax(nodes.mailbox['e'], dim=1)
        h = self.conv(alpha * nodes.mailbox['z'])
        return {'h': h}
    def forward(self, g,h):
        z = self.fc(h)
        g.ndata['z'] = z
        g.apply_edges(self.edge_attention)
        g.update_all(self.message_func, self.reduce_func)
        return g.ndata.pop('h')

class CNN(th.nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.convrow = th.nn.Sequential(
            th.nn.Conv2d(in_channels=1,
                        out_channels=1,
                        kernel_size=(2,self.in_dim)),           
            th.nn.BatchNorm2d(num_features=1),
            th.nn.ReLU(),
        )
        self.convcol = th.nn.Sequential(
            th.nn.Conv2d(in_channels=1,
                         out_channels=1,
                         kernel_size=(self.max_neighs, 1)),
            th.nn.BatchNorm2d(num_features=1),
            th.nn.ReLU(),                                
        )
    def reset_parameters(self):
        gain = nn.init.calculate_gain('relu')
        nn.init.xavier_normal_(self.convrow, gain=gain)
        nn.init.xavier_normal_(self.convcol, gain=gain)

    def forward(self, feats):
        feats = feats.unsqueeze(1)              
        featsrow = self.convrow(feats)          
        featsrow = np.squeeze(featsrow)      

        featscol = self.convcol(feats)        
        featscol = np.squeeze(featscol)      
       
        feats = th.cat((featsrow,featscol),dim=1)    
        return feats """

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from model import GATClassifier  # Your provided model
from dataset import MRIDataset, create_dataframe, stratified_patient_split
from torch_geometric.loader import DataLoader  # ✅ Use this, not torch.utils.data.DataLoader


# ---------------------
# Hyperparameters
# ---------------------
BATCH_SIZE = 64
NUM_EPOCHS = 1
LEARNING_RATE = 0.001
IN_CHANNELS = 3           # From features: mean, var, area
HIDDEN_CHANNELS = 64
NUM_CLASSES = 3

# ---------------------
# Device
# ---------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------
# Prepare Dataset
# ---------------------
directory = "./Data"
df = create_dataframe(directory)
df_train, df_test = stratified_patient_split(df, test_size=0.2)

train_dataset = MRIDataset(df_train)
test_dataset = MRIDataset(df_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ---------------------
# Model Setup
# ---------------------
model = GATClassifier(
    in_channels=IN_CHANNELS,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=NUM_CLASSES
).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

# ---------------------
# Training Loop
# ---------------------
true = []
pred = []

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for data in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        data = data.to(device)
        optimizer.zero_grad()

        out = model(data.x, data.edge_index, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(out, 1)
        
        true.append(data.y.cpu().numpy())
        pred.append(predicted.cpu().numpy())

        correct += (predicted == data.y).sum().item()
        total += data.y.size(0)

    acc = 100. * correct / total
    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Accuracy: {acc:.2f}%")


Epoch 1/1:   0%|          | 2/1089 [00:37<5:50:48, 19.36s/it]

In [2]:
from dataset import create_dataframe, stratified_patient_split, MRISeqDataset, custom_collate

directory = './Data'
df = create_dataframe(directory)
df_new, df_test = stratified_patient_split(df, test_size=0.5)
df_train, df_validation = stratified_patient_split(df_new, test_size=0.2)

train_dataset = MRISeqDataset(df_train, 30)
validation_dataset = MRISeqDataset(df_validation, 30)
test_dataset = MRISeqDataset(df_test, 30)

In [ ]:
from torch.utils.data import DataLoader
import torch
import numpy as np

BATCH_SIZE = 1
NUM_EPOCHS = 5
LEARNING_RATE = 0.001
HIDDEN_CHANNELS = 64
NUM_CLASSES = 3

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

class_weights = 1/(df['target'].value_counts()/len(df))
loss_weights = torch.tensor([
    class_weights['Non Demented'],
    class_weights['Very mild Dementia'],
    class_weights['Mild Dementia']
]).float().to(device)

In [4]:
from torch.utils.data import WeightedRandomSampler


In [8]:

label_map = {
    'Non Demented': 0,
    'Very mild Dementia': 1,
    'Mild Dementia': 2
}
targets = df_train.groupby(['patient', 'mri', 'scan']).first()['target'].map(label_map).values
class_sample_counts = np.bincount(targets)
weights = 1. / class_sample_counts
sample_weights = weights[targets]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, shuffle=True, collate_fn=custom_collate)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=custom_collate)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=custom_collate)

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from LSTM_model import LSTMModel

# Initialize model, optimizer, loss function
model = LSTMModel(in_channels=3,  # You used 3 node features: mean, var, area
                  hidden_channels=64,
                  lsmt_hidden=64,
                  num_classes=3).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(weight=loss_weights)

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch + 1}/{NUM_EPOCHS} ---")

    # TRAINING PHASE
    model.train()
    train_loss = 0
    correct_train = 0
    total_train = 0

    for sequences, labels in tqdm(train_loader, desc="Training"):
        labels = labels.to(device)
        optimizer.zero_grad()

        outputs = model(sequences)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        correct_train += predicted.eq(labels).sum().item()
        total_train += labels.size(0)

    avg_train_loss = train_loss / len(train_loader)
    train_acc = 100. * correct_train / total_train
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")

    # VALIDATION PHASE
    model.eval()
    val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for sequences, labels in tqdm(validation_loader, desc="Validation"):
            labels = labels.to(device)
            outputs = model(sequences)

            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            correct_val += predicted.eq(labels).sum().item()
            total_val += labels.size(0)

    avg_val_loss = val_loss / len(validation_loader)
    val_acc = 100. * correct_val / total_val
    print(f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")



--- Epoch 1/1 ---


Training: 100%|██████████| 568/568 [1:30:14<00:00,  9.53s/it]


Train Loss: 0.6491 | Train Acc: 79.40%


Validation: 100%|██████████| 140/140 [21:13<00:00,  9.09s/it]

Val Loss: 0.9852 | Val Acc: 66.43%


In [6]:
print(df_train['target'].value_counts(normalize=True))
print(df_validation['target'].value_counts(normalize=True))

target
Non Demented          0.794014
Very mild Dementia    0.163732
Mild Dementia         0.042254
Name: proportion, dtype: Float64
target
Non Demented          0.664286
Very mild Dementia    0.221429
Mild Dementia         0.114286
Name: proportion, dtype: Float64


#### Test

In [4]:
import torch.nn.functional as F

def train(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for sequences, labels in dataloader:
        # Move data to device
        sequences = [[g.to(device) for g in seq] for seq in sequences]
        labels = labels.to(device)

        # Forward pass
        outputs = model(sequences)  # shape: [batch_size, num_classes]
        loss = F.cross_entropy(outputs, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Logging
        total_loss += loss.item() * len(labels)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += len(labels)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc


In [ ]:
@torch.no_grad()
def validation(model, dataloader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    for sequences, labels in dataloader:
        sequences = [[g.to(device) for g in seq] for seq in sequences]
        labels = labels.to(device)

        outputs = model(sequences)
        loss = F.cross_entropy(outputs, labels)

        total_loss += loss.item() * len(labels)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += len(labels)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc


In [ ]:
# Model setup
from LSTM_model import GAT_LSTM_Model

model = GAT_LSTM_Model(in_channels=3, hidden_channels=64, lstm_hidden=128, num_classes=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train(model, train_loader, optimizer, device)
    val_loss, val_acc = validation(model, test_loader, device)

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
